In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "mps"
print(torch.backends.mps.is_available())

True


In [ ]:
ds = load_dataset("cais/mmlu", "high_school_physics", split="test")
row = ds[0]
print(row["question"])
print(row["choices"])
print(row["answer"])

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

high_school_physics/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 33.0kB            

high_school_physics/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

high_school_physics/validation-00000-of-(…): reconstructing file:   0%|          |  0.00B / 7.96kB            

high_school_physics/validation-00000-of-(…): downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating test split:   0%|          | 0/151 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

The plates of a capacitor are charged to a potential difference of 5 V. If the capacitance is 2 mF, what is the charge on the positive plate?
['0.005 C', '0.01 C', '0.02 C', '0.5 C']
1


In [4]:
model_name = "allenai/OLMo-2-0425-1B"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).to(device)
model.eval()  # evaluation mode, no training

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

W1124 21:40:06.763000 33111 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors.index.json:   0%|          | 0.00/14.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Olmo2ForCausalLM(
  (model): Olmo2Model(
    (embed_tokens): Embedding(100352, 2048, padding_idx=100277)
    (layers): ModuleList(
      (0-15): 16 x Olmo2DecoderLayer(
        (self_attn): Olmo2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Olmo2RMSNorm((2048,), eps=1e-06)
          (k_norm): Olmo2RMSNorm((2048,), eps=1e-06)
        )
        (mlp): Olmo2MLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_layernorm): Olmo2RMSNorm((2048,), eps=1e-06

In [5]:
letters = ["A", "B", "C", "D"]

def build_prompt(row):
    text = row["question"] + "\n"
    for letter, choice in zip(letters, row["choices"]):
        text += f"{letter}. {choice}\n"
    text += "Answer:"
    return text

prompt = build_prompt(row)
print(prompt)

The plates of a capacitor are charged to a potential difference of 5 V. If the capacitance is 2 mF, what is the charge on the positive plate?
A. 0.005 C
B. 0.01 C
C. 0.02 C
D. 0.5 C
Answer:


In [ ]:
letter_ids = []
for letter in letters:
    ids = tok.encode(" " + letter, add_special_tokens=False)
    print(letter, ids)
    letter_ids.append(ids[0])

A [362]
B [426]
C [356]
D [423]


In [7]:
inputs = tok(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

next_token_scores = out.logits[0, -1]
letter_scores = next_token_scores[letter_ids].float()
probs = torch.softmax(letter_scores, dim=-1)

for letter, p in zip(letters, probs.tolist()):
    print(f"{letter}: {p:.3f}")
print("Sum:", probs.sum().item())
print("Model picks:", letters[probs.argmax().item()], "| Correct:", letters[row["answer"]])

A: 0.171
B: 0.220
C: 0.437
D: 0.171
Sum: 1.0
Model picks: C | Correct: B
